# Rail Transit Accessibility Analysis

This notebook calculates rail transit accessibility metrics for urban areas, including walkable catchment areas around stations and population coverage statistics.

## Prerequisites

- Run `calc_urban_area.ipynb` first to generate the urban area boundary and population grid
- Ensure the GeoPackage files exist in the `data/` directory

## Methodology Overview

### 1. Data Loading

- Load urban area polygon from `calc_urban_area.ipynb` output
- Auto-detect optimal projected CRS for the city (national grids preferred, fallback to UTM)

### 2. Rail Network Query

- Query OpenStreetMap for rail infrastructure (subway, rail, light rail, tram, monorail)
- Cache results to GeoPackage for faster subsequent runs

### 3. Station Extraction

- Query OSM for station/stop/halt nodes within the urban area
- Filter to point geometries within the urban boundary

### 4. Walking Network & Catchment Calculation

- Fetch walkable street network from OSM
- **For large cities (>600 km²)**: Uses station-centric loading to reduce memory usage
- Compute multi-source isochrone using Dijkstra's algorithm with virtual super-source
- Buffer reachable street edges to create realistic walkable catchment polygon

### 5. Population Coverage Analysis

- Spatial join population grid cells with catchment area
- Calculate percentage of urban population within walking distance of rail transit

## Memory Optimization for Large Cities

The notebook includes automatic optimizations for large cities like London:

| Urban Area Size | Strategy                | Typical Memory Usage |
| --------------- | ----------------------- | -------------------- |
| < 600 km²       | Full network loading    | 4-12 GB              |
| > 600 km²       | Station-centric buffers | 2-6 GB               |

For very large cities, the walking network is loaded only around station buffers rather than the entire urban area, reducing memory by 70-90%.

## Outputs

- `data/rail_lines.gpkg`: Cached rail network edges and station points
- `data/station_isochrones.gpkg`: Walkable catchment polygons and covered population cells
- `data/{city}_walkable_area_map.html`: Interactive Folium map visualization

## Configuration Parameters

Key parameters in the Configuration cell:

- `WALK_TIME_MINUTES`: Walking time threshold (default: 15 min)
- `WALK_SPEED_KPH`: Walking speed assumption (default: 4.5 km/h)
- `TARGET_CRS`: Set to `None` for auto-detection, or specify manually


In [1]:
import geopandas as gpd
import osmnx as ox
import numpy as np
import networkx as nx
import pandas as pd
import fiona
import folium
from folium.plugins import MarkerCluster
from sklearn.cluster import DBSCAN
import matplotlib.pyplot as plt
from IPython.display import Image, display
from pathlib import Path
from shapely.geometry import Point, Polygon, MultiPolygon, GeometryCollection, LineString, box as shapely_box
from shapely.ops import unary_union
from shapely.wkb import dumps as wkb_dumps, loads as wkb_loads
from shapely.prepared import prep

%matplotlib inline
print(f"OSMnx version: {ox.__version__}")

OSMnx version: 2.0.2


In [2]:
# --- Helper Functions ---


def force_2d(geom):
    """Force geometry to 2D by dropping Z coordinate."""
    if geom is None or geom.is_empty:
        return geom
    try:
        return wkb_loads(wkb_dumps(geom, output_dimension=2))
    except Exception:
        return geom


def explode_to_polygons(geom):
    """Extract all Polygon geometries from any geometry type."""
    if geom is None or geom.is_empty:
        return []
    if isinstance(geom, Polygon):
        return [geom]
    if isinstance(geom, MultiPolygon):
        return [p for p in geom.geoms if isinstance(p, Polygon) and not p.is_empty]
    if isinstance(geom, GeometryCollection):
        result = []
        for g in geom.geoms:
            result.extend(explode_to_polygons(g))
        return result
    return []


def clean_and_union_geometries(gdf):
    """
    Clean geometries and create a union from a GeoDataFrame.
    Returns a single Polygon or MultiPolygon in the GDF's CRS.
    """
    # Filter valid geometries
    gdf = gdf[gdf.geometry.notna() & ~gdf.geometry.is_empty].copy()

    # Fix invalid geometries and force 2D
    gdf.geometry = gdf.geometry.apply(lambda g: force_2d(g.buffer(0)) if not g.is_valid else force_2d(g))

    # Extract all polygons
    all_polys = []
    for geom in gdf.geometry:
        all_polys.extend(explode_to_polygons(geom))

    if not all_polys:
        return None

    # Use unary_union (much faster than pairwise)
    result = unary_union(all_polys)

    # Ensure result is Polygon/MultiPolygon
    if isinstance(result, (Polygon, MultiPolygon)):
        return result
    elif isinstance(result, GeometryCollection):
        polys = explode_to_polygons(result)
        return unary_union(polys) if polys else None
    return None


def get_optimal_crs(city_boundary_gdf, city_name=None):
    """
    Automatically determine the optimal projected CRS for a city.

    Strategy:
    1. Check national/regional CRS preferences (lookup table)
    2. Fall back to UTM zone based on city centroid

    Returns:
    - EPSG code as string (e.g., "EPSG:32633")
    """
    # Lookup table for preferred national/regional CRS
    # These are often more accurate than UTM for specific countries
    national_crs = {
        # Norway
        "Oslo": "EPSG:25833",  # ETRS89 / UTM zone 33N (modern standard for Norway)
        "Bergen": "EPSG:25832",  # ETRS89 / UTM zone 32N
        "Trondheim": "EPSG:25833",
        "Stavanger": "EPSG:25832",
        # Sweden
        "Stockholm": "EPSG:3006",  # SWEREF99 TM (national grid)
        "Gothenburg": "EPSG:3006",
        "Malmö": "EPSG:3006",
        # Denmark
        "Copenhagen": "EPSG:25832",  # ETRS89 / UTM zone 32N
        # Finland
        "Helsinki": "EPSG:3067",  # ETRS89 / TM35FIN(E,N)
        # UK
        "London": "EPSG:27700",  # OSGB 1936 / British National Grid
        "Manchester": "EPSG:27700",
        "Edinburgh": "EPSG:27700",
        # France
        "Paris": "EPSG:2154",  # RGF93 / Lambert-93
        "Lyon": "EPSG:2154",
        "Marseille": "EPSG:2154",
        # Germany
        "Berlin": "EPSG:25833",  # ETRS89 / UTM zone 33N
        "Munich": "EPSG:25832",  # ETRS89 / UTM zone 32N
        "Hamburg": "EPSG:25832",
        # Netherlands
        "Amsterdam": "EPSG:28992",  # Amersfoort / RD New
        # Spain
        "Madrid": "EPSG:25830",  # ETRS89 / UTM zone 30N
        "Barcelona": "EPSG:25831",  # ETRS89 / UTM zone 31N
        # Italy
        "Rome": "EPSG:32633",  # WGS 84 / UTM zone 33N
        "Milan": "EPSG:32632",  # WGS 84 / UTM zone 32N
        # USA
        "New York": "EPSG:32618",  # WGS 84 / UTM zone 18N
        "Los Angeles": "EPSG:32611",  # WGS 84 / UTM zone 11N
        "Chicago": "EPSG:32616",  # WGS 84 / UTM zone 16N
        # Canada
        "Toronto": "EPSG:32617",  # WGS 84 / UTM zone 17N
        "Vancouver": "EPSG:32610",  # WGS 84 / UTM zone 10N
    }

    # Extract city name from "City, Country" format if needed
    if city_name:
        city_short = city_name.split(",")[0].strip()
        if city_short in national_crs:
            print(f"  Using preferred CRS for {city_short}: {national_crs[city_short]}")
            return national_crs[city_short]

    # Fall back to UTM zone calculation
    print("  Calculating optimal UTM zone from city centroid...")

    # Ensure we're in WGS84 for centroid calculation
    if city_boundary_gdf.crs != "EPSG:4326":
        city_wgs84 = city_boundary_gdf.to_crs("EPSG:4326")
    else:
        city_wgs84 = city_boundary_gdf

    # Get centroid
    centroid = city_wgs84.geometry.centroid.iloc[0]
    lon = centroid.x
    lat = centroid.y

    # Calculate UTM zone from longitude
    # UTM zones are 6 degrees wide, starting at -180°
    utm_zone = int((lon + 180) / 6) + 1

    # Determine hemisphere (North or South)
    hemisphere = "N" if lat >= 0 else "S"

    # EPSG codes for UTM zones:
    # North hemisphere: 32600 + zone (WGS84) or 25800 + zone (ETRS89 for Europe)
    # South hemisphere: 32700 + zone (WGS84)

    # Use ETRS89 for Europe (recommended for most European countries)
    if -10 <= lon <= 40 and 35 <= lat <= 72:  # Rough Europe bounds
        epsg_code = f"EPSG:{25800 + utm_zone}"
        print(f"  European location detected - using ETRS89 / UTM zone {utm_zone}N: {epsg_code}")
    else:
        # Use WGS84 UTM for rest of world
        if hemisphere == "N":
            epsg_code = f"EPSG:{32600 + utm_zone}"
        else:
            epsg_code = f"EPSG:{32700 + utm_zone}"
        print(f"  Using WGS 84 / UTM zone {utm_zone}{hemisphere}: {epsg_code}")

    return epsg_code


def get_map_center(gdf_or_geom, fallback_crs="EPSG:4326"):
    """Get map center [lat, lon] from a GeoDataFrame or geometry."""
    try:
        if isinstance(gdf_or_geom, gpd.GeoDataFrame):
            gdf_4326 = gdf_or_geom.to_crs("EPSG:4326") if gdf_or_geom.crs != "EPSG:4326" else gdf_or_geom
            centroid = gdf_4326.union_all().centroid
        else:
            centroid = gdf_or_geom.centroid
        return [float(centroid.y), float(centroid.x)]
    except Exception:
        if isinstance(gdf_or_geom, gpd.GeoDataFrame):
            bounds = gdf_or_geom.to_crs("EPSG:4326").total_bounds
        else:
            bounds = gdf_or_geom.bounds
        return [(bounds[1] + bounds[3]) / 2, (bounds[0] + bounds[2]) / 2]


def create_rail_style_function():
    """Return a style function for rail lines with consistent colors."""
    rail_colors = {
        "subway": "#0077B6",  # Deep Blue
        "rail": "#00B4D8",  # Medium Teal
        "tram": "#F4A261",  # Vibrant Orange
        "light_rail": "#9B5DE5",  # Purple
        "monorail": "#E63946",  # Red
    }
    return lambda x: {
        "color": rail_colors.get(x["properties"].get("rail_type"), "#ADB5BD"),
        "weight": 2.5,
        "opacity": 0.8,
    }


def add_legend_to_map(folium_map, title, colors_dict):
    """Add a custom legend to a folium map."""
    legend_html = f"""
    <div style="position: fixed; bottom: 50px; left: 50px; z-index: 1000;
                background-color: white; padding: 10px; border-radius: 5px;
                border: 2px solid grey; font-size: 12px;">
        <strong>{title}</strong><br>
    """
    for label, color in colors_dict.items():
        legend_html += f'<i style="background:{color}; width:12px; height:12px; display:inline-block; margin-right:5px;"></i> {label}<br>'
    legend_html += "</div>"
    folium_map.get_root().html.add_child(folium.Element(legend_html))


def spatial_filter_by_polygon(gdf, polygon, how="centroid"):
    """
    Efficiently filter GeoDataFrame by polygon containment.

    Parameters:
        gdf: GeoDataFrame to filter
        polygon: Polygon/MultiPolygon to filter by
        how: "centroid" (point-in-polygon) or "intersects"

    Returns filtered GeoDataFrame (copy).
    """
    if gdf.empty or polygon is None:
        return gdf.copy()

    # Ensure same CRS
    polygon_gdf = gpd.GeoDataFrame(geometry=[polygon], crs=gdf.crs)

    if how == "centroid":
        # Create centroids and do spatial join
        temp = gdf.copy()
        temp["_orig_geom"] = temp.geometry
        temp.geometry = temp.geometry.centroid
        result = gpd.sjoin(temp, polygon_gdf, predicate="within", how="inner")
        result.geometry = result["_orig_geom"]
        result = result.drop(columns=["_orig_geom", "index_right"])
    else:
        result = gpd.sjoin(gdf, polygon_gdf, predicate="intersects", how="inner")
        result = result.drop(columns=["index_right"], errors="ignore")

    return result.copy()

In [3]:
# --- Configuration ---

city_name = "Stockholm, Sweden"  # e.g., "Paris, France", "Stockholm, Sweden", "London, England"
city_short_name = city_name.split(",")[0].lower()

# Directories and paths
DATA_DIR = Path("./data")
DATA_DIR.mkdir(parents=True, exist_ok=True)

URBAN_AREA_GPKG = DATA_DIR / "urban_areas.gpkg"
RAIL_LINES_GPKG = DATA_DIR / "rail_lines.gpkg"
WALKING_NETWORKS_DIR = DATA_DIR / "walkable_networks"
ISOCHRONES_GPKG = DATA_DIR / "station_isochrones.gpkg"

# CRS settings
# Set to None for auto-detection, or specify manually (e.g., "EPSG:25833")
TARGET_CRS = None  # Auto-detect optimal CRS based on city location
# TARGET_CRS = "EPSG:25833"  # Uncomment to use specific CRS
WGS84 = "EPSG:4326"

# Walking parameters
WALK_TIME_MINUTES = 15
WALK_SPEED_KPH = 4.5
WALK_SPEED_MPS = WALK_SPEED_KPH * 1000 / 3600
WALK_DISTANCE_METERS = WALK_TIME_MINUTES * 60 * WALK_SPEED_MPS
STREET_BUFFER_M = 25.0  # Buffer around streets for catchment area

# Rail types to query
RAIL_TYPES = ["rail", "subway", "light_rail", "tram", "monorail"]

# Visualization colors
RAIL_COLORS = {
    "subway": "#0077B6",
    "rail": "#00B4D8",
    "tram": "#F4A261",
    "light_rail": "#9B5DE5",
    "monorail": "#E63946",
}

print(f"Configuration loaded for: {city_name}")
print(f"Walking distance: {WALK_DISTANCE_METERS:.0f}m ({WALK_TIME_MINUTES} min @ {WALK_SPEED_KPH} km/h)")

Configuration loaded for: Stockholm, Sweden
Walking distance: 1125m (15 min @ 4.5 km/h)


In [4]:
# --- Load Urban Area ---


def find_urban_area_layer(gpkg_path, city_short):
    """Find the best matching urban area layer in a GeoPackage."""
    try:
        layers = fiona.listlayers(gpkg_path)
    except Exception as e:
        print(f"Error reading {gpkg_path}: {e}")
        return None

    # Priority order for layer name patterns
    patterns = [
        f"urban_area_{city_short}_growing_refined",
        f"urban_area_{city_short}growing_refined",
        f"urban_area_{city_short}_growing",
        f"urban_area_{city_short}growing",
        f"urban_area_{city_short}_dbscan",
    ]

    for pattern in patterns:
        if pattern in layers:
            return pattern

    # Fallback: any layer with 'urban' and city name
    for layer in layers:
        if "urban" in layer.lower() and city_short in layer.lower():
            return layer

    return layers[0] if layers else None


# Load urban area
urban_area_layer = find_urban_area_layer(URBAN_AREA_GPKG, city_short_name)

if urban_area_layer is None:
    raise FileNotFoundError(f"No suitable layer found in {URBAN_AREA_GPKG}")

print(f"Loading urban area from layer: '{urban_area_layer}'")
urban_area_gdf = gpd.read_file(URBAN_AREA_GPKG, layer=urban_area_layer)
print(f"  CRS: {urban_area_gdf.crs}, Features: {len(urban_area_gdf)}")

# Auto-detect optimal CRS if not specified
if TARGET_CRS is None:
    print("Auto-detecting optimal CRS...")
    TARGET_CRS = get_optimal_crs(urban_area_gdf, city_name)
    print(f"  Selected CRS: {TARGET_CRS}")
else:
    print(f"  Using specified CRS: {TARGET_CRS}")

# Create unified polygon in WGS84 for OSM queries
urban_area_gdf_4326 = urban_area_gdf.to_crs(WGS84)
urban_area_polygon_4326 = clean_and_union_geometries(urban_area_gdf_4326)

if urban_area_polygon_4326 is None:
    raise ValueError("Failed to create valid urban area polygon")

print(f"  Urban area polygon created: {type(urban_area_polygon_4326).__name__}")

# Quick visualization
urban_area_gdf.explore(style_kwds={"fillOpacity": 0.3})

Loading urban area from layer: 'urban_area_stockholm_growing_refined'
  CRS: EPSG:4326, Features: 1
Auto-detecting optimal CRS...
  Using preferred CRS for Stockholm: EPSG:3006
  Selected CRS: EPSG:3006
  Urban area polygon created: MultiPolygon


In [5]:
# --- Query Rail Network ---

rail_lines_layer = f"rail_lines_{city_short_name}"


def query_rail_network(polygon, rail_types):
    """Query OSM for rail networks within a polygon."""
    combined_graph = nx.MultiDiGraph()

    for rail_type in rail_types:
        try:
            G = ox.graph_from_polygon(
                polygon,
                retain_all=False,
                truncate_by_edge=True,
                simplify=True,
                custom_filter=f'["railway"~"{rail_type}"]["service"!~"yard"]',
            )
            # Tag edges with rail type
            for _, _, _, data in G.edges(keys=True, data=True):
                data["rail_type"] = rail_type
            combined_graph = nx.compose(combined_graph, G)
            print(f"  {rail_type}: {len(G.edges)} edges")
        except Exception as e:
            print(f"  {rail_type}: Error - {e}")

    return combined_graph


# Try loading from cache first
gdf_edges = None
try:
    available_layers = fiona.listlayers(RAIL_LINES_GPKG)
    if rail_lines_layer in available_layers:
        gdf_edges = gpd.read_file(RAIL_LINES_GPKG, layer=rail_lines_layer)
        print(f"Loaded {len(gdf_edges)} cached rail edges from '{rail_lines_layer}'")
except Exception:
    pass

# Query OSM if no cache
if gdf_edges is None or gdf_edges.empty:
    print(f"Querying OSM for rail networks in {city_name}...")
    graph = query_rail_network(urban_area_polygon_4326, RAIL_TYPES)

    if graph.edges:
        _, gdf_edges = ox.graph_to_gdfs(graph)
        print(f"Total: {len(gdf_edges)} edges")

        # Save to cache
        gdf_edges.to_file(RAIL_LINES_GPKG, layer=rail_lines_layer, driver="GPKG")
        print(f"Saved to {RAIL_LINES_GPKG}, layer '{rail_lines_layer}'")
    else:
        print("Warning: No rail edges found")
        gdf_edges = gpd.GeoDataFrame(columns=["geometry", "rail_type"], crs=WGS84)

# Ensure correct CRS
if gdf_edges.crs is None:
    gdf_edges = gdf_edges.set_crs(WGS84)

display(gdf_edges.head(3))

Querying OSM for rail networks in Stockholm, Sweden...
  rail: 1248 edges
  subway: 837 edges
  light_rail: 218 edges
  tram: 95 edges
  monorail: Error - No data elements in server response. Check query location/filters and log.
Total: 2398 edges
Saved to data\rail_lines.gpkg, layer 'rail_lines_stockholm'


osmid bridge maxspeed              name  oneway  \
u       v          key                                                        
7524094 3830367600 0    379709846    yes       80  Västra stambanan   False   
        3830367616 0    115621236    yes       80  Västra stambanan   False   
        3830367605 0    379709850    yes       80               NaN   False   

                       reversed     length rail_type  \
u       v          key                                 
7524094 3830367600 0      False  74.611640      rail   
        3830367616 0       True  46.680356      rail   
        3830367605 0       True  36.392296      rail   

                                                                 geometry  \
u       v          key                                                      
7524094 3830367600 0    LINESTRING (18.06194 59.32706, 18.06286 59.32658)   
        3830367616 0    LINESTRING (18.06194 59.32706, 18.06166 59.327...   
        3830367605 0    LINESTRING (18.06194 59.32706, 18.06205 59.327...   

                          service tunnel access highway  
u       v          key                                   
7524094 3830367600 0          NaN    NaN    NaN     NaN  
        3830367616 0          NaN    NaN    NaN     NaN  
        3830367605 0    crossover    NaN    NaN     NaN

In [6]:
# --- Visualize Rail Network ---

if not gdf_edges.empty:
    m = urban_area_gdf.explore(style_kwds={"fillOpacity": 0.1, "color": "gray"}, name="Urban Area")

    gdf_edges.explore(
        m=m,
        column="rail_type",
        categorical=True,
        cmap=list(RAIL_COLORS.values()),
        legend=True,
        legend_kwds={"title": "Rail Type"},
        style_kwds={"weight": 3},
        name="Rail Lines",
    )

    folium.LayerControl().add_to(m)
    m
else:
    print("No rail edges to visualize")

In [7]:
# --- Query Rail Stations and Stops ---

stations_layer = f"rail_stations_{city_short_name}"
station_tags = {
    "railway": ["station", "stop", "halt"],
    "public_transport": "station",
}


def query_stations_from_osm(polygon, tags):
    """Query stations from OSM, with bbox fallback."""
    try:
        return ox.features_from_polygon(polygon, tags)
    except Exception as e:
        print(f"Polygon query failed: {e}")

        # Fallback to bbox
        minx, miny, maxx, maxy = polygon.bounds
        try:
            return ox.features_from_polygon(shapely_box(minx, miny, maxx, maxy), tags)
        except Exception as e2:
            print(f"Bbox fallback failed: {e2}")

    # Final fallback: direct Overpass query
    try:
        import overpy

        api = overpy.Overpass()
        minx, miny, maxx, maxy = polygon.bounds
        query = f"""
        [out:json][timeout:60];
        (node["railway"~"station|stop|halt"]({miny},{minx},{maxy},{maxx}););
        out body;
        """
        result = api.query(query)
        rows = [
            {
                "osmid": int(n.id),
                "name": n.tags.get("name"),
                "railway": n.tags.get("railway"),
                "public_transport": n.tags.get("public_transport"),
                "geometry": Point(float(n.lon), float(n.lat)),
            }
            for n in result.nodes
        ]
        return gpd.GeoDataFrame(rows, geometry="geometry", crs=WGS84)
    except Exception as e3:
        print(f"Overpass fallback failed: {e3}")
        return gpd.GeoDataFrame(columns=["geometry"], crs=WGS84)


print(f"Querying stations within {city_name} urban area...")
gdf_stops = query_stations_from_osm(urban_area_polygon_4326, station_tags)

if gdf_stops.empty:
    print("No stations found")
    gdf_stops_points = gpd.GeoDataFrame(columns=["geometry"], crs=WGS84)
else:
    # Ensure WGS84 CRS
    if gdf_stops.crs is None or gdf_stops.crs != WGS84:
        gdf_stops = gdf_stops.set_crs(WGS84, allow_override=True)

    # Filter to points only
    gdf_stops_points = gdf_stops[gdf_stops.geometry.notna() & (gdf_stops.geometry.geom_type == "Point")].copy()

    # Filter to within urban area using spatial join (faster than .apply)
    if not gdf_stops_points.empty:
        urban_poly_gdf = gpd.GeoDataFrame(geometry=[urban_area_polygon_4326], crs=WGS84)
        gdf_stops_points = gpd.sjoin(gdf_stops_points, urban_poly_gdf, predicate="within", how="inner").drop(
            columns=["index_right"], errors="ignore"
        )

    print(f"Found {len(gdf_stops_points)} station points within urban area")

# Save stations
if not gdf_stops_points.empty:
    save_gdf = gdf_stops_points.copy()
    save_gdf["city_name"] = city_name
    save_gdf["city_short"] = city_short_name
    save_gdf.to_file(RAIL_LINES_GPKG, layer=stations_layer, driver="GPKG")
    print(f"Saved to {RAIL_LINES_GPKG}, layer '{stations_layer}'")

Querying stations within Stockholm, Sweden urban area...
Found 265 station points within urban area
Saved to data\rail_lines.gpkg, layer 'rail_stations_stockholm'


In [8]:
# --- Visualize Rail Network with Stations (Clustered) ---

if not gdf_edges.empty or not gdf_stops_points.empty:
    map_center = get_map_center(urban_area_gdf)
    m = folium.Map(location=map_center, zoom_start=11, tiles="CartoDB positron")

    # Add rail lines with styling
    if not gdf_edges.empty:
        gdf_plot = gdf_edges.to_crs(WGS84) if gdf_edges.crs != WGS84 else gdf_edges
        folium.GeoJson(
            gdf_plot,
            name="Rail Lines",
            style_function=create_rail_style_function(),
            tooltip=folium.GeoJsonTooltip(fields=["rail_type", "name"], aliases=["Type:", "Name:"], localize=True),
        ).add_to(m)

    # Add stations with marker clustering (much faster for many points)
    if not gdf_stops_points.empty:
        cluster = MarkerCluster(name="Stations")
        stops_plot = gdf_stops_points.to_crs(WGS84) if gdf_stops_points.crs != WGS84 else gdf_stops_points

        for _, row in stops_plot.iterrows():
            name = row.get("name", "Station") or "Station"
            rail_type = row.get("railway", "unknown")
            folium.CircleMarker(
                location=[row.geometry.y, row.geometry.x],
                radius=6,
                popup=f"<b>{name}</b><br>Type: {rail_type}",
                color="darkred",
                fill=True,
                fillColor="red",
                fillOpacity=0.7,
            ).add_to(cluster)

        cluster.add_to(m)

    # Add legend
    add_legend_to_map(m, "Rail Types", RAIL_COLORS)
    folium.LayerControl().add_to(m)
    display(m)
else:
    print("No rail data to visualize")

In [ ]:
# --- Load Population Grid ---


def find_population_grid_layer(gpkg_path, city_short):
    """Find the best matching population grid layer in a GeoPackage."""
    try:
        layers = fiona.listlayers(gpkg_path)
    except Exception as e:
        print(f"Error reading {gpkg_path}: {e}")
        return None

    # Priority order for layer name patterns (prefer refined versions)
    patterns = [
        f"urban_cells_{city_short}_growing_refined",
        f"urban_cells_{city_short}growing_refined",
        f"urban_cells_{city_short}_growing",
        f"urban_cells_{city_short}growing",
        f"urban_cells_{city_short}",
    ]

    for pattern in patterns:
        if pattern in layers:
            return pattern

    # Fallback: any layer with 'urban_cells' and city name
    for layer in layers:
        if "urban_cells" in layer.lower() and city_short in layer.lower():
            return layer

    return None


# Find and load population grid
urban_cells_layer = find_population_grid_layer(URBAN_AREA_GPKG, city_short_name)

if urban_cells_layer is None:
    print(f"Warning: No population grid layer found for '{city_short_name}' in {URBAN_AREA_GPKG}")
    print("Population coverage statistics will not be available.")
    print("Run calc_urban_area.ipynb first to generate the population grid.")
    urban_cells_proj = None
    total_urban_population = 0
else:
    try:
        print(f"Loading population grid from layer '{urban_cells_layer}'...")
        urban_cells_gdf = gpd.read_file(URBAN_AREA_GPKG, layer=urban_cells_layer)
        urban_cells_proj = urban_cells_gdf.to_crs(TARGET_CRS)

        total_urban_population = urban_cells_proj["population"].sum()
        print(f"Loaded {len(urban_cells_proj):,} cells")
        print(f"Total urban population: {total_urban_population:,.0f}")

    except Exception as e:
        print(f"Error loading population grid: {e}")
        urban_cells_proj = None
        total_urban_population = 0

# Station Catchment Analysis

Calculate the walkable area around all rail stations using network-based isochrones rather than simple circular buffers. This provides a more realistic estimate of accessibility.


In [ ]:
# --- Calculate Walkable Catchment Area ---

# Verify prerequisites
if gdf_stops_points.empty:
    raise ValueError("Missing prerequisite data: stops not loaded")

# --- Configuration for Large Cities ---
# For very large cities (London, NYC, etc.), loading the entire walking network
# can exceed available RAM. Set USE_STATION_BUFFERS=True to load network
# only around stations, dramatically reducing memory usage.

# Estimate urban area size to auto-select strategy
urban_area_km2 = urban_area_polygon_4326.area * 111**2  # rough degrees to km conversion
print(f"Urban area size: ~{urban_area_km2:.0f} km²")

# Auto-select strategy based on area size (London ~1,500 km², Stockholm ~200 km²)
# With 32GB RAM, ~500 km² is a reasonable threshold for full network loading
USE_STATION_BUFFERS = urban_area_km2 > 600  # Switch to buffer mode for large cities
STATION_BUFFER_RADIUS = WALK_DISTANCE_METERS * 1.5  # Extra margin for network connectivity

if USE_STATION_BUFFERS:
    print(f"⚠️  Large urban area detected - using station-centric network loading")
    print(f"   (Loading {STATION_BUFFER_RADIUS:.0f}m buffer around each station)")
else:
    print(f"   Using full urban area network loading")

# --- Load or Fetch Walking Network ---
WALKING_NETWORKS_DIR.mkdir(parents=True, exist_ok=True)
walk_graph_path = WALKING_NETWORKS_DIR / f"{city_short_name}_walk_proj_{TARGET_CRS.replace(':', '')}.graphml"

G_walk_proj = None

if USE_STATION_BUFFERS:
    # MEMORY-EFFICIENT MODE: Load network only around stations
    import gc

    print("Fetching walking network around stations...")
    gdf_stops_4326 = gdf_stops_points.to_crs(WGS84) if gdf_stops_points.crs != WGS84 else gdf_stops_points

    # Create buffer around all stations (in projected CRS for accurate distance)
    gdf_stops_proj_temp = gdf_stops_points.to_crs(TARGET_CRS)
    station_buffers = gdf_stops_proj_temp.buffer(STATION_BUFFER_RADIUS)
    combined_buffer = unary_union(station_buffers)

    # Convert back to WGS84 for OSM query
    buffer_gdf = gpd.GeoDataFrame(geometry=[combined_buffer], crs=TARGET_CRS).to_crs(WGS84)
    buffer_polygon = buffer_gdf.geometry.iloc[0]

    print(f"  Station buffer area: {combined_buffer.area / 1_000_000:.1f} km² (vs {urban_area_km2:.0f} km² full area)")

    try:
        G_walk_unproj = ox.graph_from_polygon(buffer_polygon, network_type="walk", simplify=True, truncate_by_edge=True)
        G_walk_proj = ox.project_graph(G_walk_unproj, to_crs=TARGET_CRS)
        del G_walk_unproj
        gc.collect()
        print(f"  Network: {len(G_walk_proj.nodes):,} nodes, {len(G_walk_proj.edges):,} edges")
    except Exception as e:
        print(f"Error fetching station-buffered network: {e}")
        print("Falling back to full urban area (may require significant memory)...")
        USE_STATION_BUFFERS = False  # Fall through to standard loading

if not USE_STATION_BUFFERS and G_walk_proj is None:
    # STANDARD MODE: Load entire urban area network
    try:
        print(f"Loading cached walking network from {walk_graph_path}...")
        G_walk_proj = ox.load_graphml(walk_graph_path)
        print(f"  Loaded: {len(G_walk_proj.nodes):,} nodes, {len(G_walk_proj.edges):,} edges")
    except FileNotFoundError:
        print("Cache not found, fetching from OSM...")
    except Exception as e:
        print(f"Error loading cache: {e}")

    if G_walk_proj is None:
        print("Fetching walkable street network (this may take a while for large cities)...")
        G_walk_unproj = ox.graph_from_polygon(
            urban_area_polygon_4326, network_type="walk", simplify=True, truncate_by_edge=True
        )
        G_walk_proj = ox.project_graph(G_walk_unproj, to_crs=TARGET_CRS)
        ox.save_graphml(G_walk_proj, filepath=walk_graph_path)
        print(f"  Saved to {walk_graph_path}")
        print(f"  Network: {len(G_walk_proj.nodes):,} nodes, {len(G_walk_proj.edges):,} edges")

In [ ]:
# --- Compute Multi-Source Isochrone ---


def compute_walkable_isochrone(G, stop_coords_proj, cutoff_meters, buffer_m=25.0, chunk_size=50000):
    """
    Compute walkable catchment area using multi-source Dijkstra.

    Parameters:
        G: NetworkX graph (projected)
        stop_coords_proj: array of (x, y) coordinates in graph CRS
        cutoff_meters: walking distance cutoff
        buffer_m: buffer around reachable edges
        chunk_size: number of edges to buffer/union at once (memory optimization)

    Returns:
        Polygon/MultiPolygon of walkable area
    """
    import gc

    # Convert to undirected for symmetric walking - MEMORY OPTIMIZATION:
    # Instead of copying all data, only copy essential attributes
    if isinstance(G, (nx.MultiDiGraph, nx.DiGraph)):
        Gu = nx.Graph()
        Gu.graph.update(G.graph)

        # Pre-allocate node coordinates for faster access later
        node_coords = {}
        for n, nd in G.nodes(data=True):
            if "x" in nd and "y" in nd:
                node_coords[n] = (nd["x"], nd["y"])
                Gu.add_node(n, x=nd["x"], y=nd["y"])

        # Add edges with minimum length (avoid storing duplicate edge data)
        for u, v, data in G.edges(data=True):
            w = data.get("length", 1.0)
            if Gu.has_edge(u, v):
                if w < Gu[u][v]["length"]:
                    Gu[u][v]["length"] = w
            else:
                Gu.add_edge(u, v, length=w)
    else:
        Gu = G
        node_coords = {n: (nd["x"], nd["y"]) for n, nd in G.nodes(data=True) if "x" in nd and "y" in nd}

    # Find nearest nodes to all stops
    stop_nodes = ox.nearest_nodes(Gu, stop_coords_proj[:, 0], stop_coords_proj[:, 1])
    unique_nodes = list(set(stop_nodes))
    print(f"  Mapped {len(stop_coords_proj)} stops to {len(unique_nodes)} unique graph nodes")

    # Multi-source Dijkstra via virtual super-source
    super_src = "__super__"
    G_aug = Gu.copy()
    G_aug.add_node(super_src)
    for node in unique_nodes:
        if node in G_aug:
            G_aug.add_edge(super_src, node, length=0.0)

    # Run single-source Dijkstra from super-source
    print(f"  Running Dijkstra (cutoff={cutoff_meters}m)...")
    dist_map = nx.single_source_dijkstra_path_length(G_aug, super_src, cutoff=cutoff_meters, weight="length")
    dist_map.pop(super_src, None)
    reachable = set(dist_map.keys())
    print(f"  Reachable nodes: {len(reachable):,}")

    # Free memory from augmented graph
    del G_aug
    gc.collect()

    # Collect reachable edges as coordinate tuples (more memory efficient than LineStrings)
    edge_coords = []
    for u, v in Gu.edges():
        if u in reachable and v in reachable:
            if u in node_coords and v in node_coords:
                edge_coords.append((node_coords[u], node_coords[v]))

    print(f"  Reachable edges: {len(edge_coords):,}")

    if not edge_coords:
        # Fallback: buffer reachable nodes
        pts = [Point(node_coords[n]) for n in reachable if n in node_coords]
        return unary_union([p.buffer(50) for p in pts]) if pts else None

    # MEMORY OPTIMIZATION: Process edges in chunks for large networks
    if len(edge_coords) > chunk_size:
        print(f"  Large network detected - processing in chunks of {chunk_size:,}...")
        chunk_results = []

        for i in range(0, len(edge_coords), chunk_size):
            chunk = edge_coords[i : i + chunk_size]
            # Create LineStrings and buffer in one pass
            buffered_chunk = [LineString([c[0], c[1]]).buffer(buffer_m) for c in chunk]
            chunk_union = unary_union(buffered_chunk)
            chunk_results.append(chunk_union)

            # Progress indicator for very large networks
            if len(edge_coords) > 100000 and (i + chunk_size) % 100000 == 0:
                print(f"    Processed {min(i + chunk_size, len(edge_coords)):,} / {len(edge_coords):,} edges...")

            # Free memory
            del buffered_chunk
            gc.collect()

        # Final union of chunk results
        print(f"  Merging {len(chunk_results)} chunks...")
        result = unary_union(chunk_results)
        del chunk_results
        gc.collect()
    else:
        # Original approach for smaller networks
        edge_geoms = [LineString([c[0], c[1]]) for c in edge_coords]
        buffered = [geom.buffer(buffer_m) for geom in edge_geoms]
        result = unary_union(buffered)

    # Ensure valid result
    if not result.is_valid:
        result = result.buffer(0)

    return result


# Compute isochrone
print(f"Computing {WALK_TIME_MINUTES}-min walking catchment from {len(gdf_stops_points)} stops...")
gdf_stops_proj = gdf_stops_points.to_crs(TARGET_CRS)
stop_coords = np.column_stack([gdf_stops_proj.geometry.x, gdf_stops_proj.geometry.y])

walkable_area = compute_walkable_isochrone(G_walk_proj, stop_coords, WALK_DISTANCE_METERS, STREET_BUFFER_M)

if walkable_area is None or walkable_area.is_empty:
    print("Warning: Could not compute walkable area")
else:
    print(f"  Walkable area: {walkable_area.area / 1_000_000:.2f} km²")

In [ ]:
# --- Calculate Population Within Walking Distance ---

population_within_walk = 0
coverage_percentage = 0
cells_within_walk = None

if walkable_area is not None and not walkable_area.is_empty and total_urban_population > 0:
    print("Computing population coverage...")

    # Use efficient spatial join instead of .apply().within()
    cells_within_walk = spatial_filter_by_polygon(urban_cells_proj, walkable_area, how="centroid")

    population_within_walk = cells_within_walk["population"].sum()
    coverage_percentage = (population_within_walk / total_urban_population) * 100

    print(f"\n=== Coverage Statistics ===")
    print(f"  Cells within walking distance: {len(cells_within_walk):,}")
    print(f"  Population covered: {population_within_walk:,.0f}")
    print(f"  Coverage: {coverage_percentage:.1f}% of urban population")
    print(f"  Walking time: {WALK_TIME_MINUTES} minutes")
    print(f"  Walking distance: {WALK_DISTANCE_METERS:.0f} m")
else:
    print("Cannot calculate coverage: missing walkable area or population data")

In [ ]:
# --- Visualize Catchment Area with Population ---

if walkable_area is not None and not walkable_area.is_empty:
    print("Creating catchment visualization...")
    map_center = get_map_center(urban_area_gdf)
    viz_map = folium.Map(location=map_center, zoom_start=11, tiles="CartoDB positron")

    # Add walkable catchment area
    walkable_gdf = gpd.GeoDataFrame(geometry=[walkable_area], crs=TARGET_CRS).to_crs(WGS84)
    folium.GeoJson(
        walkable_gdf,
        name=f"{WALK_TIME_MINUTES}-Min Walk Catchment",
        style_function=lambda x: {"fillColor": "#3388ff", "color": "#3388ff", "weight": 1, "fillOpacity": 0.25},
    ).add_to(viz_map)

    # Add population cells with choropleth coloring
    if cells_within_walk is not None and not cells_within_walk.empty:
        cells_viz = cells_within_walk.to_crs(WGS84).copy()
        if "centroid" in cells_viz.columns:
            cells_viz = cells_viz.drop(columns=["centroid"])

        folium.Choropleth(
            geo_data=cells_viz,
            data=cells_viz,
            columns=[cells_viz.index.name or "index", "population"],
            key_on="feature.id",
            fill_color="YlOrRd",
            fill_opacity=0.6,
            line_opacity=0.1,
            legend_name="Population per Cell",
            name="Population Cells",
        ).add_to(viz_map)

    # Add stations with marker cluster
    if not gdf_stops_points.empty:
        cluster = MarkerCluster(name="Stations", show=True)
        stops_viz = gdf_stops_points.to_crs(WGS84)
        for _, row in stops_viz.iterrows():
            name = row.get("name", "Station") or "Station"
            folium.CircleMarker(
                location=[row.geometry.y, row.geometry.x],
                radius=5,
                popup=name,
                color="red",
                fill=True,
                fillOpacity=0.8,
            ).add_to(cluster)
        cluster.add_to(viz_map)

    # Add statistics box
    stats_html = f"""
    <div style="position: fixed; top: 10px; right: 10px; z-index: 1000;
                background: white; padding: 12px; border-radius: 5px;
                border: 2px solid #333; font-size: 13px; max-width: 220px;">
        <strong>Coverage Statistics</strong><br>
        <hr style="margin: 5px 0;">
        Walking time: {WALK_TIME_MINUTES} min<br>
        Distance: {WALK_DISTANCE_METERS:.0f} m<br>
        Stations: {len(gdf_stops_points)}<br>
        <hr style="margin: 5px 0;">
        Population: {population_within_walk:,.0f}<br>
        <strong>Coverage: {coverage_percentage:.1f}%</strong>
    </div>
    """
    viz_map.get_root().html.add_child(folium.Element(stats_html))

    folium.LayerControl().add_to(viz_map)

    # Save and display
    output_html = DATA_DIR / f"{city_short_name}_walkable_area_map.html"
    viz_map.save(str(output_html))
    print(f"  Saved to {output_html}")
    display(viz_map)
else:
    print("No walkable area to visualize")

In [ ]:
# --- Save Isochrone Data ---

if walkable_area is not None and not walkable_area.is_empty:
    isochrone_layer = f"isochrones_{city_short_name}_{WALK_TIME_MINUTES}min"

    # Create isochrone GeoDataFrame
    isochrone_gdf = gpd.GeoDataFrame(
        {
            "city_name": [city_name],
            "city_short": [city_short_name],
            "walk_time_minutes": [WALK_TIME_MINUTES],
            "walk_speed_kph": [WALK_SPEED_KPH],
            "walk_distance_m": [WALK_DISTANCE_METERS],
            "stop_count": [len(gdf_stops_points)],
            "area_sqkm": [walkable_area.area / 1_000_000],
            "population_covered": [population_within_walk],
            "coverage_percentage": [coverage_percentage],
            "geometry": [walkable_area],
        },
        crs=TARGET_CRS,
    ).to_crs(WGS84)

    # Save isochrone polygon
    isochrone_gdf.to_file(ISOCHRONES_GPKG, layer=isochrone_layer, driver="GPKG")
    print(f"Saved isochrone to {ISOCHRONES_GPKG}, layer '{isochrone_layer}'")

    # Save individual stop buffers
    if not gdf_stops_points.empty:
        buffers_layer = f"stop_buffers_{city_short_name}_{WALK_TIME_MINUTES}min"
        stop_buffers = gdf_stops_proj.copy()
        stop_buffers.geometry = stop_buffers.geometry.buffer(WALK_DISTANCE_METERS)
        stop_buffers["walk_time_minutes"] = WALK_TIME_MINUTES
        stop_buffers["walk_distance_m"] = WALK_DISTANCE_METERS
        stop_buffers.to_crs(WGS84).to_file(ISOCHRONES_GPKG, layer=buffers_layer, driver="GPKG")
        print(f"Saved stop buffers to layer '{buffers_layer}'")

    # Save population cells within walking distance
    if cells_within_walk is not None and not cells_within_walk.empty:
        pop_layer = f"population_within_walk_{city_short_name}_{WALK_TIME_MINUTES}min"
        cells_save = cells_within_walk.copy()
        if "centroid" in cells_save.columns:
            cells_save = cells_save.drop(columns=["centroid"])
        cells_save["walk_time_minutes"] = WALK_TIME_MINUTES
        cells_save["city_name"] = city_name
        cells_save.to_crs(WGS84).to_file(ISOCHRONES_GPKG, layer=pop_layer, driver="GPKG")
        print(f"Saved population cells to layer '{pop_layer}'")
        print(f"  Total population saved: {cells_save['population'].sum():,.0f}")

    print("\nIsochrone analysis complete!")
else:
    print("No isochrone geometry to save")